# 15 — Compare: LongBench (kvpress vs vLLM)

Side-by-side comparison of KV cache compression on
[LongBench](https://github.com/THUDM/LongBench) tasks from
notebooks 11 (kvpress) and 12 (vLLM).

Two LongBench tasks:
- **gov_report** — summarization, scored with ROUGE-L
- **hotpotqa** — multi-hop QA, scored with F1

Both frameworks test **KeyDiffPress**-based compression with two
decoding strategies:
- **full_replacement** — full KV cache replacement after prefill
- **filtering** — token filtering during decoding

Visualizations:
1. Per-task line charts (one per LongBench task)
2. Summary table

**Note**: kvpress (notebook 11) stores ROUGE-L on a 0–1 scale,
while vLLM (notebook 12) stores it on a 0–100 scale. This notebook
normalizes all scores to 0–100.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

ALGORITHMS = ['full_replacement', 'filtering']
SHORT_NAMES = {
    'full_replacement': 'Full Replacement',
    'filtering': 'Filtering',
    'no_press': 'No compression',
}
TASK_LABELS = {
    'gov_report': 'gov_report — ROUGE-L (%)',
    'hotpotqa': 'hotpotqa — F1 (%)',
}

## 1. Load Results

In [ ]:
def load_longbench_metrics(path, framework):
    with open(path) as f:
        raw = json.load(f)
    rows = []
    for key, scores in raw.items():
        parts = key.split('__')
        press = parts[0]
        ratio = float(parts[1])
        task = parts[2]
        metric_name = list(scores.keys())[0]
        score = scores[metric_name]
        rows.append({
            'framework': framework,
            'press': press,
            'compression_ratio': ratio,
            'task': task,
            'metric': metric_name,
            'score': score,
        })
    return pd.DataFrame(rows)

kvpress_df = load_longbench_metrics('results/kvpress_longbench/metrics.json', 'kvpress')
vllm_df = load_longbench_metrics('results/vllm_longbench/metrics.json', 'vllm')

# Normalize: kvpress stores ROUGE-L on 0-1 scale, vLLM on 0-100
mask = (kvpress_df['metric'] == 'rougeL') & (kvpress_df['score'] <= 1.0)
kvpress_df.loc[mask, 'score'] = kvpress_df.loc[mask, 'score'] * 100

df = pd.concat([kvpress_df, vllm_df], ignore_index=True)

print(f'Loaded {len(kvpress_df)} kvpress + {len(vllm_df)} vllm = {len(df)} total')
print(f'Tasks: {sorted(df["task"].unique())}')
print(f'Algorithms: {sorted(df["press"].unique())}')

## 2. Score vs Compression Ratio

One panel per LongBench task, with kvpress and vLLM lines overlaid.
- **gov_report**: ROUGE-L (summarization quality)
- **hotpotqa**: F1 (QA accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = {'kvpress': 'tab:blue', 'vllm': 'tab:orange'}
styles = {'full_replacement': '-', 'filtering': '--'}
markers = {'full_replacement': 'o', 'filtering': 's'}

for ax, task in zip(axes, ['gov_report', 'hotpotqa']):
    task_df = df[df['task'] == task]

    for fw in ['kvpress', 'vllm']:
        for algo in ALGORITHMS:
            sub = task_df[(task_df['framework'] == fw) & (task_df['press'] == algo)]
            sub = sub.sort_values('compression_ratio')
            if sub.empty:
                continue
            ax.plot(
                sub['compression_ratio'], sub['score'],
                marker=markers[algo], linewidth=2,
                color=colors[fw], linestyle=styles[algo],
                label=f'{fw} — {SHORT_NAMES[algo]}',
            )

    for fw in ['kvpress', 'vllm']:
        baseline = task_df[(task_df['framework'] == fw) & (task_df['press'] == 'no_press')]
        if not baseline.empty:
            score = baseline['score'].values[0]
            ax.axhline(
                score, color=colors[fw], linestyle=':', linewidth=1.5, alpha=0.5,
                label=f'{fw} — No compression ({score:.1f})',
            )

    ax.set_xlabel('Compression Ratio')
    ax.set_ylabel(TASK_LABELS[task])
    ax.set_title(f'LongBench: {task}')
    ax.legend(fontsize=9)

fig.suptitle('LongBench: kvpress vs vLLM — Score by Compression Ratio', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig('results/compare_longbench_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Summary

In [ ]:
summary = (
    df.groupby(['framework', 'press', 'compression_ratio', 'task'])
    .agg(score=('score', 'first'))
    .round(2)
)
print(summary.to_string())